# Data Scientist (итерация 1)

# Data Scientist Report — Fake Job Postings

**Бизнес-задача:** бинарная классификация мошеннических вакансий (`fraudulent`). Приоритет — F1 и recall класса 1 при контроле precision (≥0.5 желательно).

**План:**
1. Загрузка `cleaned.csv`, стратифицированный split train/val/test.
2. Feature engineering: word-counts по текстовым полям, бинарные флаги пустых полей (company_profile_empty и т.п.), TF-IDF + TruncatedSVD по объединённому тексту.
3. Baseline: LogisticRegression (class_weight='balanced'), RandomForest, GradientBoosting. Сравнение по F1 на val.
4. Tuning лучшей модели через RandomizedSearchCV (cv=3, scoring='f1').
5. Финальная оценка на test + оптимизация threshold по F1 на val.
6. Сохранение модели, метрик, сравнение с предыдущим best.
7. Self-critique.

In [ ]:
import pandas as pd
import numpy as np
import os, json, joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, precision_recall_curve,
                             average_precision_score)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FIGS = []
RND = 42

CLEANED_PATH = '/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv'
FEATURES_PATH = '/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/features.csv'
MODEL_PATH = '/Users/iuriipostnii/Desktop/ГП3/gp3/data/memory/best_model.pkl'
METRICS_PATH = '/Users/iuriipostnii/Desktop/ГП3/gp3/data/memory/best_metrics.json'

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FEATURES_PATH), exist_ok=True)

DF = pd.read_csv(CLEANED_PATH)
print('Shape:', DF.shape)
print('Target balance:\n', DF['fraudulent'].value_counts())
print('Positive rate:', DF['fraudulent'].mean().round(4))
print('Columns (first 30):', list(DF.columns)[:30])
DF.head(2)

Shape: (17880, 75)
Target balance:
 fraudulent
0    17014
1      866
Name: count, dtype: int64
Positive rate: 0.0484
Columns (first 30): ['job_id', 'title', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'fraudulent', 'employment_type_Contract', 'employment_type_Full-time', 'employment_type_Other', 'employment_type_Part-time', 'employment_type_Temporary', 'required_experience_Associate', 'required_experience_Director', 'required_experience_Entry level', 'required_experience_Executive', 'required_experience_Internship', 'required_experience_Mid-Senior level', 'required_experience_Not Applicable', 'required_education_Associate Degree', "required_education_Bachelor's Degree", 'required_education_Certification', 'required_education_Doctorate', 'required_education_High School or equivalent', "required_education_Master's Degree", 'required_education_Professional', 'required_education_Some College Coursework Completed']


## Train/val/test split (стратификация по target)

Разделение 70/15/15 со стратификацией по `fraudulent`, чтобы доля позитивов сохранилась во всех сплитах.

In [ ]:
TARGET = 'fraudulent'
TEXT_COLS = [c for c in ['title', 'description', 'requirements', 'benefits', 'company_profile'] if c in DF.columns]
print('Text columns:', TEXT_COLS)

X = DF.drop(columns=[TARGET])
y = DF[TARGET].astype(int)

# 70 / 15 / 15
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RND)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765,  # 0.15 / 0.85 ≈ 0.1765
    stratify=y_trainval, random_state=RND)

print(f'Train: {X_train.shape}, pos={y_train.mean():.4f}')
print(f'Val:   {X_val.shape}, pos={y_val.mean():.4f}')
print(f'Test:  {X_test.shape}, pos={y_test.mean():.4f}')

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'scale_pos_weight (neg/pos) = {scale_pos_weight:.2f}')

Text columns: ['title', 'description', 'requirements', 'benefits', 'company_profile']
Train: (12515, 74), pos=0.0484
Val:   (2683, 74), pos=0.0485
Test:  (2682, 74), pos=0.0485
scale_pos_weight (neg/pos) = 19.65


## Feature engineering

Добавляем:
1. **Word counts** по каждой текстовой колонке — `*_wc`. По EDA короткие описания чаще фродовые.
2. **Empty flags** — `is_*_empty` (особенно `is_company_profile_empty`, lift ≈4.2x).
3. **Char-length** по description и requirements.
4. **Суммарная длина текста** (`total_text_wc`) и **средняя длина слова** в description.
5. **TF-IDF** по конкатенации text-колонок (max_features=20000, ngram=(1,2)) → **TruncatedSVD(n_components=64)** — плотные текстовые эмбеддинги `svd_0..svd_63`.

Все FE считаются отдельно для train/val/test; TF-IDF и SVD фитятся **только на train**.

In [ ]:
def add_text_features(df_in, text_cols):
    df = df_in.copy()
    for c in text_cols:
        s = df[c].fillna('').astype(str)
        df[f'{c}_wc'] = s.str.split().apply(len)
        df[f'{c}_charlen'] = s.str.len()
        df[f'is_{c}_empty'] = (s.str.strip() == '').astype(int)
    wc_cols = [f'{c}_wc' for c in text_cols]
    df['total_text_wc'] = df[wc_cols].sum(axis=1)
    # средняя длина слова в description
    if 'description' in text_cols:
        desc = df['description'].fillna('').astype(str)
        df['description_avg_word_len'] = desc.str.len() / (desc.str.split().apply(len).replace(0, 1))
    return df

X_train_fe = add_text_features(X_train, TEXT_COLS)
X_val_fe   = add_text_features(X_val,   TEXT_COLS)
X_test_fe  = add_text_features(X_test,  TEXT_COLS)

new_feats = [c for c in X_train_fe.columns if c not in X_train.columns]
print(f'Добавлено {len(new_feats)} новых фичей:')
for f in new_feats:
    print(' -', f)

# ---- TF-IDF + SVD по конкатенации текстов ----
def concat_text(df):
    return (df[TEXT_COLS].fillna('').astype(str).agg(' '.join, axis=1))

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3,
                        sublinear_tf=True, strip_accents='unicode')
svd = TruncatedSVD(n_components=64, random_state=RND)

Xtr_tfidf = tfidf.fit_transform(concat_text(X_train_fe))
Xva_tfidf = tfidf.transform(concat_text(X_val_fe))
Xte_tfidf = tfidf.transform(concat_text(X_test_fe))

Xtr_svd = svd.fit_transform(Xtr_tfidf)
Xva_svd = svd.transform(Xva_tfidf)
Xte_svd = svd.transform(Xte_tfidf)
print('SVD explained variance ratio sum:', svd.explained_variance_ratio_.sum().round(3))

svd_cols = [f'svd_{i}' for i in range(Xtr_svd.shape[1])]
Xtr_svd_df = pd.DataFrame(Xtr_svd, columns=svd_cols, index=X_train_fe.index)
Xva_svd_df = pd.DataFrame(Xva_svd, columns=svd_cols, index=X_val_fe.index)
Xte_svd_df = pd.DataFrame(Xte_svd, columns=svd_cols, index=X_test_fe.index)

# Собираем финальную числовую матрицу: дропаем raw text, добавляем SVD
def finalize(df_fe, svd_df):
    num = df_fe.drop(columns=TEXT_COLS, errors='ignore')
    # оставляем только числовые
    num = num.select_dtypes(include=[np.number])
    return pd.concat([num, svd_df], axis=1)

X_train_final = finalize(X_train_fe, Xtr_svd_df)
X_val_final   = finalize(X_val_fe,   Xva_svd_df)
X_test_final  = finalize(X_test_fe,  Xte_svd_df)

print('Final feature shapes:', X_train_final.shape, X_val_final.shape, X_test_final.shape)

# Сохранение полного фичевого датасета
all_features = pd.concat([
    X_train_final.assign(**{TARGET: y_train, 'split': 'train'}),
    X_val_final.assign(  **{TARGET: y_val,   'split': 'val'}),
    X_test_final.assign( **{TARGET: y_test,  'split': 'test'}),
], axis=0)
all_features.to_csv(FEATURES_PATH, index=False)
print('Saved features to', FEATURES_PATH, 'shape=', all_features.shape)

Добавлено 17 новых фичей:
 - title_wc
 - title_charlen
 - is_title_empty
 - description_wc
 - description_charlen
 - is_description_empty
 - requirements_wc
 - requirements_charlen
 - is_requirements_empty
 - benefits_wc
 - benefits_charlen
 - is_benefits_empty
 - company_profile_wc
 - company_profile_charlen
 - is_company_profile_empty
 - total_text_wc
 - description_avg_word_len
SVD explained variance ratio sum: 0.254
Final feature shapes: (12515, 150) (2683, 150) (2682, 150)
Saved features to /Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/features.csv shape= (17880, 152)


### Быстрая проверка сигналов новых фичей

Проверяем mean target rate по децилям `description_wc` и lift для `is_company_profile_empty`.

In [ ]:
tmp = X_train_final.copy()
tmp['target'] = y_train.values
if 'is_company_profile_empty' in tmp.columns:
    lift = tmp.groupby('is_company_profile_empty')['target'].mean()
    print('Fraud rate by is_company_profile_empty:\n', lift.round(4))
    print('Lift:', (lift.get(1,0) / max(lift.get(0,1e-9), 1e-9)).round(2))

if 'description_wc' in tmp.columns:
    tmp['desc_bucket'] = pd.qcut(tmp['description_wc'], q=10, duplicates='drop')
    by_bucket = tmp.groupby('desc_bucket')['target'].agg(['mean','count'])
    print('\nFraud rate by description_wc decile:')
    print(by_bucket.round(4))
    fig = px.bar(by_bucket.reset_index().astype({'desc_bucket': str}),
                 x='desc_bucket', y='mean', title='Fraud rate by description_wc decile (train)')
    FIGS.append(fig)

Fraud rate by is_company_profile_empty:
 is_company_profile_empty
0    0.0186
1    0.1797
Name: target, dtype: float64
Lift: 9.65

Fraud rate by description_wc decile:
                   mean  count
desc_bucket                   
(-0.001, 40.0]   0.0390   1258
(40.0, 73.0]     0.0979   1267
(73.0, 97.0]     0.0661   1241
(97.0, 119.0]    0.0602   1279
(119.0, 143.0]   0.0441   1224
(143.0, 169.0]   0.0192   1251
(169.0, 203.0]   0.0253   1265
(203.0, 246.0]   0.0382   1229
(246.0, 321.0]   0.0344   1251
(321.0, 1268.0]  0.0592   1250


## Baseline-модели

Три модели с учётом дисбаланса:
- **LogisticRegression** (`class_weight='balanced'`) с масштабированием.
- **RandomForestClassifier** (`class_weight='balanced'`).
- **GradientBoostingClassifier** (sample_weight по классу).

Метрики считаем на val. Threshold=0.5 для первого прохода, позже подберём по F1.

In [ ]:
def eval_model(model, X_tr, y_tr, X_va, y_va, name, sample_weight=None):
    if sample_weight is not None:
        model.fit(X_tr, y_tr, sample_weight=sample_weight)
    else:
        model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_va)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        'model': name,
        'f1': f1_score(y_va, pred),
        'precision': precision_score(y_va, pred, zero_division=0),
        'recall': recall_score(y_va, pred),
        'roc_auc': roc_auc_score(y_va, proba),
        'pr_auc': average_precision_score(y_va, proba),
        '_model': model,
        '_proba': proba,
    }

results = []

# 1. Logistic Regression + StandardScaler
logreg = Pipeline([
    ('sc', StandardScaler(with_mean=True)),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=2000,
                              C=1.0, solver='liblinear', random_state=RND)),
])
results.append(eval_model(logreg, X_train_final, y_train, X_val_final, y_val, 'LogReg'))

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_leaf=2,
                            class_weight='balanced', n_jobs=-1, random_state=RND)
results.append(eval_model(rf, X_train_final, y_train, X_val_final, y_val, 'RandomForest'))

# 3. Gradient Boosting (через sample_weight для учёта дисбаланса)
sw = np.where(y_train == 1, scale_pos_weight, 1.0)
gb = GradientBoostingClassifier(n_estimators=250, max_depth=3, learning_rate=0.1, random_state=RND)
results.append(eval_model(gb, X_train_final, y_train, X_val_final, y_val, 'GradientBoosting', sample_weight=sw))

results_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in results])
print(results_df.round(4).to_string(index=False))

fig = px.bar(results_df.melt(id_vars='model', value_vars=['f1','precision','recall','roc_auc','pr_auc']),
             x='model', y='value', color='variable', barmode='group',
             title='Baseline metrics on VAL')
FIGS.append(fig)

best_idx = results_df['f1'].idxmax()
best_name = results_df.loc[best_idx, 'model']
print(f'\nBest baseline by F1 on val: {best_name}')

           model     f1  precision  recall  roc_auc  pr_auc
          LogReg 0.4264     0.2825  0.8692   0.9477  0.6071
    RandomForest 0.7890     0.9773  0.6615   0.9912  0.9316
GradientBoosting 0.8013     0.7126  0.9154   0.9904  0.9290

Best baseline by F1 on val: GradientBoosting


## Подбор гиперпараметров для лучшей baseline-модели

RandomizedSearchCV (cv=3, scoring='f1') на train+val объединённо. Для GradientBoosting варьируем `n_estimators`, `max_depth`, `learning_rate`, `min_samples_leaf`, `subsample`.

In [ ]:
X_tv = pd.concat([X_train_final, X_val_final], axis=0)
y_tv = pd.concat([y_train, y_val], axis=0)
sw_tv = np.where(y_tv == 1, scale_pos_weight, 1.0)

if best_name == 'GradientBoosting':
    base = GradientBoostingClassifier(random_state=RND)
    param_dist = {
        'n_estimators': [150, 250, 400, 600],
        'max_depth': [2, 3, 4, 5],
        'learning_rate': [0.03, 0.05, 0.1, 0.15],
        'min_samples_leaf': [1, 5, 20, 50],
        'subsample': [0.7, 0.85, 1.0],
    }
    fit_params = {'sample_weight': sw_tv}
elif best_name == 'RandomForest':
    base = RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=RND)
    param_dist = {
        'n_estimators': [200, 400, 600, 800],
        'max_depth': [None, 10, 20, 30],
        'min_samples_leaf': [1, 2, 5, 10],
        'max_features': ['sqrt', 'log2', 0.3],
    }
    fit_params = {}
else:
    base = Pipeline([('sc', StandardScaler()),
                     ('lr', LogisticRegression(class_weight='balanced', max_iter=3000,
                                               solver='liblinear', random_state=RND))])
    param_dist = {
        'lr__C': [0.01, 0.05, 0.1, 0.5, 1, 3, 10],
        'lr__penalty': ['l1', 'l2'],
    }
    fit_params = {}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RND)
search = RandomizedSearchCV(base, param_distributions=param_dist, n_iter=15,
                            scoring='f1', cv=cv, n_jobs=-1, random_state=RND,
                            verbose=1, refit=True)
search.fit(X_tv, y_tv, **fit_params)
print('Best params:', search.best_params_)
print('Best CV f1:', round(search.best_score_, 4))
best_estimator = search.best_estimator_

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best params: {'subsample': 1.0, 'n_estimators': 400, 'min_samples_leaf': 5, 'max_depth': 2, 'learning_rate': 0.05}
Best CV f1: 0.9312


## Финальная модель и тест

Используем best_estimator (уже refit на train+val). Подбираем threshold по F1 на val, применяем к test.

In [ ]:
# threshold tuning on val
val_proba = best_estimator.predict_proba(X_val_final)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, val_proba)
f1_arr = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
best_thr_idx = int(np.argmax(f1_arr[:-1])) if len(thr) > 0 else 0
best_threshold = float(thr[best_thr_idx]) if len(thr) > 0 else 0.5
print(f'Best threshold on val: {best_threshold:.4f} (val F1={f1_arr[best_thr_idx]:.4f})')

# test evaluation
test_proba = best_estimator.predict_proba(X_test_final)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

test_metrics = {
    'f1': float(f1_score(y_test, test_pred)),
    'precision': float(precision_score(y_test, test_pred, zero_division=0)),
    'recall': float(recall_score(y_test, test_pred)),
    'roc_auc': float(roc_auc_score(y_test, test_proba)),
    'pr_auc': float(average_precision_score(y_test, test_proba)),
    'threshold': best_threshold,
}
print('\nTEST metrics:')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

cm = confusion_matrix(y_test, test_pred)
print('\nConfusion matrix (test):\n', cm)

fig_cm = go.Figure(data=go.Heatmap(
    z=cm, x=['pred_0','pred_1'], y=['true_0','true_1'],
    text=cm, texttemplate='%{text}', colorscale='Blues'))
fig_cm.update_layout(title=f'Confusion matrix on TEST (thr={best_threshold:.3f})')
FIGS.append(fig_cm)

# PR curve
prec_t, rec_t, _ = precision_recall_curve(y_test, test_proba)
fig_pr = go.Figure()
fig_pr.add_trace(go.Scatter(x=rec_t, y=prec_t, mode='lines', name='PR test'))
fig_pr.update_layout(title=f'Precision-Recall on TEST (PR-AUC={test_metrics["pr_auc"]:.3f})',
                     xaxis_title='Recall', yaxis_title='Precision')
FIGS.append(fig_pr)

# save model (включаем best_threshold + препроцессоры)
artefact = {
    'model': best_estimator,
    'tfidf': tfidf,
    'svd': svd,
    'text_cols': TEXT_COLS,
    'feature_columns': list(X_train_final.columns),
    'threshold': best_threshold,
}
joblib.dump(artefact, MODEL_PATH)
print('Model saved to', MODEL_PATH)

Best threshold on val: 0.7811 (val F1=0.8699)

TEST metrics:
  f1: 0.7654
  precision: 0.8230
  recall: 0.7154
  roc_auc: 0.9756
  pr_auc: 0.8387
  threshold: 0.7811

Confusion matrix (test):
 [[2532   20]
 [  37   93]]
Model saved to /Users/iuriipostnii/Desktop/ГП3/gp3/data/memory/best_model.pkl


## Сравнение с предыдущим best

Если `best_metrics.json` уже существует — сравниваем по test F1. Перезаписываем только если новая модель лучше.

In [ ]:
new_record = {
    'model_name': f'{best_name}_tuned',
    'best_params': {k: (v if isinstance(v, (int,float,str,bool,type(None))) else str(v))
                    for k, v in search.best_params_.items()},
    'test_metrics': test_metrics,
    'val_f1_at_threshold': float(f1_arr[best_thr_idx]) if len(thr) > 0 else None,
    'n_features': int(X_train_final.shape[1]),
    'n_train': int(len(y_train)),
    'n_val': int(len(y_val)),
    'n_test': int(len(y_test)),
}

prev = None
if os.path.exists(METRICS_PATH):
    try:
        prev = json.load(open(METRICS_PATH))
        print('Previous best:', prev.get('model_name'), '| f1:', prev.get('test_metrics', {}).get('f1'))
    except Exception as e:
        print('Failed to read previous metrics:', e)
        prev = None
else:
    print('No previous best_metrics.json found — first run.')

prev_f1 = (prev or {}).get('test_metrics', {}).get('f1', -1)
new_f1 = test_metrics['f1']
print(f'\nNew test F1: {new_f1:.4f} | Previous test F1: {prev_f1}')

if prev is None or new_f1 > prev_f1:
    json.dump(new_record, open(METRICS_PATH, 'w'), indent=2)
    print(f'✅ Saved new best to {METRICS_PATH}')
else:
    # Всё равно обязаны записать согласно ТЗ — но сохраним лучший. Оставляем prev.
    json.dump(prev, open(METRICS_PATH, 'w'), indent=2)
    print('ℹ️ New model is NOT better. Previous best retained.')

print('\nFinal content of best_metrics.json:')
print(json.dumps(json.load(open(METRICS_PATH)), indent=2))

No previous best_metrics.json found — first run.

New test F1: 0.7654 | Previous test F1: -1
✅ Saved new best to /Users/iuriipostnii/Desktop/ГП3/gp3/data/memory/best_metrics.json

Final content of best_metrics.json:
{
  "model_name": "GradientBoosting_tuned",
  "best_params": {
    "subsample": 1.0,
    "n_estimators": 400,
    "min_samples_leaf": 5,
    "max_depth": 2,
    "learning_rate": 0.05
  },
  "test_metrics": {
    "f1": 0.7654320987654321,
    "precision": 0.8230088495575221,
    "recall": 0.7153846153846154,
    "roc_auc": 0.9755817458403665,
    "pr_auc": 0.8386692522336775,
    "threshold": 0.7810923897608004
  },
  "val_f1_at_threshold": 0.8698884758364313,
  "n_features": 150,
  "n_train": 12515,
  "n_val": 2683,
  "n_test": 2682
}


## Self-critique — что осталось для следующих итераций

**Что сделали:**
- Стратифицированный 70/15/15 split; честная оценка только на test.
- Feature engineering из текстов: word counts, char length, empty-флаги, total_text_wc, avg_word_len, TF-IDF(1,2) + TruncatedSVD(64).
- 3 baseline-модели с учётом дисбаланса; лучший — GBDT/RF/LogReg (см. таблицу).
- RandomizedSearchCV (15 iter, 3-fold CV, scoring=f1) по 4–5 гиперпараметрам.
- Threshold tuning по F1 на val; финальная оценка на test (F1, precision, recall, ROC-AUC, PR-AUC).
- Сохранены: features.csv, best_model.pkl (с tfidf+svd+threshold), best_metrics.json.

**Что не успели / гипотезы на следующую итерацию:**
1. **LightGBM / XGBoost с native scale_pos_weight** — обычно дают +2–4 п.п. F1 на таких задачах; GBDT из sklearn медленнее и без early stopping.
2. **Char-level TF-IDF** (ngram 3–5) — хорошо ловит подделанные адреса/урлы; может дополнить word-level.
3. **Target encoding через out-of-fold** для high-card колонок (`industry_freq`, `location_freq`), а также bucketized qcut-версии для LogReg.
4. **Stacking**: LogReg(TF-IDF) + LightGBM(tabular) → мета-LogReg. Обычно лучший способ выжать последние проценты.
5. **Calibration** (isotonic/Platt) для более стабильного threshold в проде.
6. **Анализ ошибок** — посмотреть FP/FN вручную; возможно, нужны фичи из URLs/emails/phones в тексте.
7. **Мониторинг дрейфа** `*_freq` фичей — распределения зависят от train; для прода нужен пересчёт/freeze.
8. **Group split по company** (если доступен стабильный идентификатор) — сейчас один работодатель мог попасть и в train, и в test, что завышает метрики.